# Real estate analysis in Paris

**Dataset:** Demandes de Valeurs Foncières (DVF) • declared land & property transaction values  
**Source:** [data.gouv.fr / DGFiP](https://www.data.gouv.fr/datasets/demandes-de-valeurs-foncieres/)  

### 1. Load Dataset

In [ ]:
import zipfile
import pandas as pd
import gdown
import os
from tqdm import tqdm

# Configuration: add or remove years here
YEARS = [2025]

FILE_IDS = {
    2025: "11RcrfNqLQ091vatGB688wH3XpA44XLsP",
    2024: "1MbgEisO5V9YnSw6-s4Dej5jdijkwwwLo",
    2023: "1RLWZPoOy2KtxKblpUY9z6OwGpn-F72xb",
    2022: "1dqAt9s5vkVNkLrptWV3QY5rm_payjAzG",
    2021: "1w-GS0zAC4viZKZhSnAysk22I6zviBu8v"
}

all_dfs = []

for YEAR in YEARS:
    file_id      = FILE_IDS.get(YEAR)
    zip_filename = f"valeursfoncieres-{YEAR}.txt.zip"

    # Download from Google Drive
    if not os.path.exists(zip_filename):
        print(f"Downloading {zip_filename}...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url, zip_filename, quiet=False)
    else:
        print(f"{zip_filename} already exists locally.")

    # Open ZIP and read with progress bar
    with zipfile.ZipFile(zip_filename) as z:
        file_name = z.namelist()[0]
        print(f"Reading {YEAR} — file inside ZIP: {file_name}")
        with z.open(file_name) as f:
            chunk_size = 100_000
            chunks = []
            with tqdm(desc=f"Loading {YEAR}", unit=" chunks") as pbar:
                reader = pd.read_csv(
                    f,
                    sep="|",
                    low_memory=False,
                    encoding="utf-8",
                    chunksize=chunk_size
                )
                for chunk in reader:
                    chunks.append(chunk)
                    pbar.update(1)

    df_year = pd.concat(chunks, ignore_index=True)
    df_year["year"] = YEAR   # tag each row with its source year
    print(f"  {YEAR}: {df_year.shape[0]:,} rows x {df_year.shape[1]} columns\n")
    all_dfs.append(df_year)

# Combine into one raw dataframe
df_raw = pd.concat(all_dfs, ignore_index=True)
print(f"Combined dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Years present: {sorted(df_raw['year'].unique())}")

Downloading...
From (original): https://drive.google.com/uc?id=11RcrfNqLQ091vatGB688wH3XpA44XLsP
From (redirected): https://drive.google.com/uc?id=11RcrfNqLQ091vatGB688wH3XpA44XLsP&confirm=t&uuid=b06c78d8-b1d1-4d70-a973-8b88f9f62655
To: /content/valeursfoncieres-2025.txt.zip
100%|██████████| 69.7M/69.7M [00:02<00:00, 32.7MB/s]


Reading 2025 — file inside ZIP: ValeursFoncieres-2025.txt


Loading 2025: 38 chunks [00:24,  1.57 chunks/s]


  2025: 3,714,829 rows x 44 columns

Combined dataset shape: 3,714,829 rows x 44 columns
Years present: [np.int64(2025)]


In [ ]:
# Snapshot of raw dataset.
# Stored for accurate summary table at the end.

snap_raw_rows    = len(df_raw)
snap_raw_cols    = df_raw.shape[1]
snap_raw_missing = int(df_raw.isna().sum().sum())

print(f"Snapshot saved — {snap_raw_rows:,} rows | {snap_raw_cols} columns | {snap_raw_missing:,} missing cells")


Snapshot saved — 3,714,829 rows | 44 columns | 90,144,608 missing cells


---
#### Initial Inspection
Before any cleaning, we examine the raw dataset to understand its structure,
data types, and the extent of missing values.

In [ ]:
# First 5 rows: check on column names and values
df_raw.head()

,Identifiant de document,Reference document,1 Articles CGI,2 Articles CGI,3 Articles CGI,4 Articles CGI,5 Articles CGI,No disposition,Date mutation,Nature mutation,...,Nombre de lots,Code type local,Type local,Identifiant local,Surface reelle bati,Nombre pieces principales,Nature culture,Nature culture speciale,Surface terrain,year
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,J,NaN,78.0,2025
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,0,3.0,Dépendance,NaN,0.0,0.0,S,NaN,133.0,2025
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,07/01/2025,Vente,...,0,1.0,Maison,NaN,111.0,5.0,S,NaN,133.0,2025
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,S,NaN,46.0,2025
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,06/01/2025,Vente,...,0,NaN,NaN,NaN,NaN,NaN,J,NaN,17.0,2025


#### Data types & non-null counts

In [ ]:
# Column names, non-null counts and inferred dtypes for every column.

df_raw.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3714829 entries, 0 to 3714828
Data columns (total 44 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   Identifiant de document     0 non-null        float64
 1   Reference document          0 non-null        float64
 2   1 Articles CGI              0 non-null        float64
 3   2 Articles CGI              0 non-null        float64
 4   3 Articles CGI              0 non-null        float64
 5   4 Articles CGI              0 non-null        float64
 6   5 Articles CGI              0 non-null        float64
 7   No disposition              3714829 non-null  int64  
 8   Date mutation               3714829 non-null  object 
 9   Nature mutation             3714829 non-null  object 
 10  Valeur fonciere             3666409 non-null  object 
 11  No voie                     2400906 non-null  float64
 12  B/T/Q                       171323 non-null   object 
 1

#### Missing values analysis

In [ ]:
# Shows the percentage of missing values per column, sorted descending.
# Columns above 50% will be dropped in Step 2.

missing_pct = df_raw.isna().mean() * 100
missing_df = (
    missing_pct
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_%"})
    .sort_values("missing_%", ascending=False)
)

print("Columns with > 50% missing values (will be dropped):")
print(missing_df[missing_df["missing_%"] > 50].to_string(index=False))
print(f"\nTotal columns with >50% missing values: {(missing_df['missing_%'] > 50).sum()}")

Columns with > 50% missing values (will be dropped):
                    column  missing_%
   Identifiant de document 100.000000
        Reference document 100.000000
            1 Articles CGI 100.000000
            2 Articles CGI 100.000000
            3 Articles CGI 100.000000
            4 Articles CGI 100.000000
            5 Articles CGI 100.000000
         Identifiant local 100.000000
Surface Carrez du 5eme lot  99.963255
Surface Carrez du 4eme lot  99.908529
                 No Volume  99.761146
                  5eme lot  99.704078
Surface Carrez du 3eme lot  99.663242
                  4eme lot  99.418170
                  3eme lot  98.290608
Surface Carrez du 2eme lot  96.918243
   Nature culture speciale  95.569406
                     B/T/Q  95.388132
        Prefixe de section  95.146075
 Surface Carrez du 1er lot  90.557600
                  2eme lot  90.331453
                   1er lot  68.996581

Total columns with >50% missing values: 22


#### Variable type classification

In [ ]:
# Check unique value counts for all categorical columns
# to inform a sensible classification threshold

cat_cols = df_raw.select_dtypes(include="object").columns

unique_counts = (
    pd.Series({col: df_raw[col].nunique() for col in cat_cols})
    .sort_values(ascending=False)
)

print("Unique value counts per categorical column:")
print(unique_counts.to_string())
print(f"\nMin : {unique_counts.min()}")
print(f"Max : {unique_counts.max()}")
print(f"Mean: {unique_counts.mean():.0f}")

Unique value counts per categorical column:
Voie                          449472
Valeur fonciere               144286
Commune                        30776
Surface Carrez du 1er lot      17654
Code voie                      16252
1er lot                        12380
Surface Carrez du 2eme lot     11592
2eme lot                        5847
Surface Carrez du 3eme lot      4156
3eme lot                        1564
Surface Carrez du 4eme lot      1241
No Volume                        725
Section                          581
Surface Carrez du 5eme lot       484
Date mutation                    361
Type de voie                     137
Nature culture speciale          125
Code departement                  99
B/T/Q                             40
Nature culture                    27
Nature mutation                    6
Type local                         4

Min : 4
Max : 449472
Mean: 31719


In [ ]:
# Variable type classification
# Classifies each column by dtype and number of unique values.
# Thresholds defined based on the actual distribution of unique values in this dataset.

categories = []

for col in df_raw.columns:
    series = df_raw[col].dropna()

    if series.dtype in ["int64", "float64"]:
        var_type = "Quantitative"
    else:
        n_unique = series.nunique()
        if n_unique == 1:
            var_type = "Constant (single value)"
        elif n_unique == 2:
            var_type = "Binary"
        elif n_unique <= 10:
            var_type = "Categorical — Low cardinality (3–10)"
        elif n_unique <= 100:
            var_type = "Categorical — Medium cardinality (11–100)"
        elif n_unique <= 1000:
            var_type = "Categorical — High cardinality (101–1000)"
        else:
            var_type = "Categorical — Very high cardinality (>1000)"

    categories.append({
        "Column":    col,
        "Dtype":     str(df_raw[col].dtype),
        "Unique":    series.nunique(),
        "Type":      var_type,
        "Missing %": round(df_raw[col].isna().mean() * 100, 1)
    })

var_type_df = pd.DataFrame(categories)
display(var_type_df)

,Column,Dtype,Unique,Type,Missing %
0,Identifiant de document,float64,0,Quantitative,100.0
1,Reference document,float64,0,Quantitative,100.0
2,1 Articles CGI,float64,0,Quantitative,100.0
3,2 Articles CGI,float64,0,Quantitative,100.0
4,3 Articles CGI,float64,0,Quantitative,100.0
5,4 Articles CGI,float64,0,Quantitative,100.0
6,5 Articles CGI,float64,0,Quantitative,100.0
7,No disposition,int64,208,Quantitative,0.0
8,Date mutation,object,361,Categorical — High cardinality (101–1000),0.0
9,Nature mutation,object,6,Categorical — Low cardinality (3–10),0.0


### 2. Filter: Keep Paris Only

The raw DVF file covers all of France (~3M+ rows).
Filter for Paris only. `Code departement` equals `'75'` (Paris).



In [ ]:
# Keep department as string and filter Paris department 75
df_raw["Code departement"] = df_raw["Code departement"].astype(str)
df_paris = df_raw[df_raw["Code departement"] == "75"].copy()

# Convert postal code to numeric and keep Paris postal codes 75001 to 75020
df_paris["Code postal"] = pd.to_numeric(df_paris["Code postal"], errors="coerce").astype("Int64")
df_paris = df_paris[(df_paris["Code postal"] >= 75001) & (df_paris["Code postal"] <= 75020)]

print(f"Rows before filter : {len(df_raw):>10,}")
print(f"Rows after filter  : {len(df_paris):>10,}")
print(f"Paris share        : {len(df_paris)/len(df_raw)*100:.1f}%")

print(sorted(df_paris["Code postal"].dropna().unique()))


Rows before filter :  3,714,829
Rows after filter  :     84,252
Paris share        : 2.3%
[np.int64(75001), np.int64(75002), np.int64(75003), np.int64(75004), np.int64(75005), np.int64(75006), np.int64(75007), np.int64(75008), np.int64(75009), np.int64(75010), np.int64(75011), np.int64(75012), np.int64(75013), np.int64(75014), np.int64(75015), np.int64(75016), np.int64(75017), np.int64(75018), np.int64(75019), np.int64(75020)]


In [ ]:
# Check: only Paris rows should remain
print("Unique values in 'Code departement' after filter:")
print(df_paris["Code departement"].unique(), "\n")

print("Unique values in 'Code postal' after filter:")
print(df_paris["Code postal"].unique())

print(f"\nTotal rows: {len(df_paris):,}")

Unique values in 'Code departement' after filter:
['75'] 

Unique values in 'Code postal' after filter:
<IntegerArray>
[75004, 75018, 75010, 75002, 75008, 75017, 75003, 75009, 75019, 75020, 75001,
 75005, 75015, 75006, 75012, 75014, 75016, 75011, 75013, 75007]
Length: 20, dtype: Int64

Total rows: 84,252


In [ ]:
# Snapshot for the cleaning  table at the end.
snap_paris_rows    = len(df_paris)
snap_paris_cols    = df_paris.shape[1]
snap_paris_missing = int(df_paris.isna().sum().sum())

print(f"Snapshot saved — {snap_paris_rows:,} rows | {snap_paris_cols} columns | {snap_paris_missing:,} missing cells")

Snapshot saved — 84,252 rows | 44 columns | 1,842,337 missing cells


---
### 3. Drop Columns with > 50% Missing Values
Many DVF columns are structural placeholders that are only filled for specific
transaction types (e.g. lot details for co-owned apartments, agricultural land
culture fields). In urban Paris these are almost always empty.

We remove any column where more than 50% of values are `NaN`.


In [ ]:
# Drop sparse columns (> 50% missing)
MISSING_THRESHOLD = 0.50

missing_ratio = df_paris.isna().mean()                              # fraction missing per column
cols_to_drop  = missing_ratio[missing_ratio > MISSING_THRESHOLD].index.tolist()
cols_to_keep  = missing_ratio[missing_ratio <= MISSING_THRESHOLD].index.tolist()

print(f"Columns before : {df_paris.shape[1]}")
print(f"Columns dropped: {len(cols_to_drop)}")
print(f"Columns kept   : {len(cols_to_keep)}")
print(f"\nDropped columns:")
for col in cols_to_drop:
    print(f"  • {col:<45} ({missing_ratio[col]*100:.1f}% missing)")

df_paris = df_paris[cols_to_keep]

Columns before : 44
Columns dropped: 23
Columns kept   : 21

Dropped columns:
  • Identifiant de document                       (100.0% missing)
  • Reference document                            (100.0% missing)
  • 1 Articles CGI                                (100.0% missing)
  • 2 Articles CGI                                (100.0% missing)
  • 3 Articles CGI                                (100.0% missing)
  • 4 Articles CGI                                (100.0% missing)
  • 5 Articles CGI                                (100.0% missing)
  • B/T/Q                                         (96.0% missing)
  • Prefixe de section                            (100.0% missing)
  • No Volume                                     (99.8% missing)
  • Surface Carrez du 1er lot                     (62.8% missing)
  • 2eme lot                                      (58.6% missing)
  • Surface Carrez du 2eme lot                    (89.0% missing)
  • 3eme lot                                      (93.0%

In [ ]:
# Snapshot for the cleaning summary table at the end
snap_drop_cols_rows    = len(df_paris)
snap_drop_cols_cols    = df_paris.shape[1]
snap_drop_cols_missing = int(df_paris.isna().sum().sum())

print(f"Snapshot saved — {snap_drop_cols_rows:,} rows | {snap_drop_cols_cols} columns | {snap_drop_cols_missing:,} missing cells")

Snapshot saved — 84,252 rows | 21 columns | 12,065 missing cells


In [ ]:
# Analyze the structure and quality of the remaining columns

print("### Variable Type Classification & Missingness Profile ###")

categories = []
for col in df_paris.columns:
    series = df_paris[col].dropna()
    n_unique = series.nunique()
    total_rows = len(df_paris)
    missing_pct = df_paris[col].isna().mean() * 100

    # Identify Data Type
    if pd.api.types.is_datetime64_any_dtype(df_paris[col]):
        var_type = "Temporal (Datetime)"
    elif n_unique == total_rows:
        var_type = "Unique Identifier (Primary Key)"
    elif pd.api.types.is_numeric_dtype(df_paris[col]):
        var_type = "Quantitative (Numeric)"
    else:
        # Categorical Logic based on Cardinality
        if n_unique == 1:
            var_type = "Constant (Single Value)"
        elif n_unique == 2:
            var_type = "Binary"
        elif n_unique <= 20:
            var_type = "Categorical (Low Cardinality)"
        elif n_unique <= 100:
            var_type = "Categorical (Medium Cardinality)"
        else:
            var_type = "Categorical (High Cardinality)"

    categories.append({
        "Column": col,
        "Dtype": str(df_paris[col].dtype),
        "Unique": n_unique,
        "Type": var_type,
        "Missing %": round(missing_pct, 1)
    })

# Convert to DataFrame for better visualization
var_type_df = pd.DataFrame(categories).sort_values("Missing %", ascending=False)

# Highlight columns that still have missing values
display(var_type_df.style.background_gradient(subset=['Missing %'], cmap='Reds'))

### Variable Type Classification & Missingness Profile ###


,Column,Dtype,Unique,Type,Missing %
14,1er lot,object,2484,Categorical (High Cardinality),11.000000
3,Valeur fonciere,object,11422,Categorical (High Cardinality),1.300000
18,Surface reelle bati,float64,644,Quantitative (Numeric),0.500000
17,Type local,object,4,Categorical (Low Cardinality),0.500000
16,Code type local,float64,4,Quantitative (Numeric),0.500000
19,Nombre pieces principales,float64,16,Quantitative (Numeric),0.500000
5,Type de voie,object,20,Categorical (Low Cardinality),0.100000
1,Date mutation,object,291,Categorical (High Cardinality),0.000000
0,No disposition,int64,5,Quantitative (Numeric),0.000000
4,No voie,float64,369,Quantitative (Numeric),0.000000


In [ ]:
#  Impact analysis: what would a global dropna() cost us?
# This cell quantifies exactly how many rows we would lose with different strategies.

rows_total     = len(df_paris)
rows_after_all = df_paris.dropna().shape[0]
pct_lost_all   = (rows_total - rows_after_all) / rows_total * 100

print("=" * 55)
print("  dropna() impact analysis")
print("=" * 55)
print(f"  Rows after Step 3 (Paris, sparse cols removed) : {rows_total:>8,}")
print(f"  Rows after global dropna() on ALL columns      : {rows_after_all:>8,}")
print(f"  Rows lost                                      : {rows_total - rows_after_all:>8,}")
print(f"  Data retained                                  : {100 - pct_lost_all:>7.1f}%")
print(f"  Data lost                                      : {pct_lost_all:>7.1f}%")
print("=" * 55)

# Per-column missing value breakdown
print("\nMissing values per column (after Step 3):\n")
missing = df_paris.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
for col, count in missing.items():
    pct = count / rows_total * 100
    bar = "█" * int(pct / 2)
    print(f"  {col:<40} {count:>6,}  ({pct:>5.1f}%)  {bar}")

  dropna() impact analysis
  Rows after Step 3 (Paris, sparse cols removed) :   84,252
  Rows after global dropna() on ALL columns      :   74,825
  Rows lost                                      :    9,427
  Data retained                                  :    88.8%
  Data lost                                      :    11.2%

Missing values per column (after Step 3):

  1er lot                                   9,242  ( 11.0%)  █████
  Valeur fonciere                           1,054  (  1.3%)  
  Surface reelle bati                         432  (  0.5%)  
  Nombre pieces principales                   432  (  0.5%)  
  Type local                                  425  (  0.5%)  
  Code type local                             425  (  0.5%)  
  Type de voie                                 53  (  0.1%)  
  No voie                                       2  (  0.0%)  


### 4. Drop Rows with Missing Values

In [ ]:
# Drop "1er lot" column. No analytical value and high share of missing values (11.2%)
df_paris = df_paris.drop(columns=["1er lot"])

# Drop rows with any remaining missing values
rows_before = len(df_paris)
df_paris = df_paris.dropna()
rows_after = len(df_paris)

print(f"Rows before dropna : {rows_before:>8,}")
print(f"Rows after dropna  : {rows_after:>8,}")
print(f"Rows lost          : {rows_before - rows_after:>8,}")
print(f"Data retained      : {rows_after / rows_before * 100:>7.1f}%")

Rows before dropna :   84,252
Rows after dropna  :   82,752
Rows lost          :    1,500
Data retained      :    98.2%


### Date Type Conversion
The column `Date mutation` is loaded as string -> convert to `datetime64`.


In [ ]:
# Convert 'Date mutation' from object to datetime
# The format in the raw file is DD/MM/YYYY.
# errors='coerce' converts unparseable values to NaT (Not a Time) instead of raising an error.
df_paris["Date mutation"] = pd.to_datetime(
    df_paris["Date mutation"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("Dtype after conversion:", df_paris["Date mutation"].dtype)
print("Number of NaT values:", df_paris["Date mutation"].isna().sum())
print("Date range:", df_paris["Date mutation"].min(), "-", df_paris["Date mutation"].max())


# Convert 'Valeur fonciere' from object to float. Required for later price analysis
# The dataset uses comma as decimal separator, but Python expects a dot as the decimal separator.
# Replace commas with dots before converting to float.

df_paris["Valeur fonciere"] = pd.to_numeric(
    df_paris["Valeur fonciere"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Dtype after conversion:", df_paris["Valeur fonciere"].dtype)
print("Number of NaN values:", df_paris["Valeur fonciere"].isna().sum())

Dtype after conversion: datetime64[ns]
Number of NaT values: 0
Date range: 2025-01-02 00:00:00 - 2025-12-31 00:00:00
Dtype after conversion: float64
Number of NaN values: 0


### 5. Post-Cleaning Validation
Verify that the cleaned dataset meets all expected conditions before
using it for API enrichment.

In [ ]:
# Check: only Paris rows remain
if df_paris["Code departement"].nunique() != 1:
    print(f"Unexpected departments found: {df_paris['Code departement'].unique()}")
else:
    print("Only Paris present.")

Only Paris present.


In [ ]:
if not df_paris["Date mutation"].dt.year.isin([2025]).all():
    print("Dates other than 2025 found")
else:
    print("All dates are within 2025.")

All dates are within 2025.


In [ ]:
# Final dtypes and shape after Steps 1–3
# Confirm data types look correct after cleaning and date conversion.

print("Current dtypes:")
print(df_paris.dtypes)
print(f"\nCurrent shape: {df_paris.shape[0]:,} rows x {df_paris.shape[1]} columns")

Current dtypes:
No disposition                        int64
Date mutation                datetime64[ns]
Nature mutation                      object
Valeur fonciere                     float64
No voie                             float64
Type de voie                         object
Code voie                            object
Voie                                 object
Code postal                           Int64
Commune                              object
Code departement                     object
Code commune                          int64
Section                              object
No plan                               int64
Nombre de lots                        int64
Code type local                     float64
Type local                           object
Surface reelle bati                 float64
Nombre pieces principales           float64
year                                  int64
dtype: object

Current shape: 82,752 rows x 20 columns


In [ ]:
# Summary statistics on the partially cleaned 2024-2025 dataset.

df_paris.describe()

,No disposition,Date mutation,Valeur fonciere,No voie,Code postal,Code commune,No plan,Nombre de lots,Code type local,Surface reelle bati,Nombre pieces principales,year
count,82752.000000,82752,8.275200e+04,82752.000000,82752.0,82752.000000,82752.000000,82752.000000,82752.000000,82752.000000,82752.000000,82752.0
mean,1.006417,2025-07-08 00:29:37.030162432,3.844312e+06,48.048845,75013.249215,113.249215,49.621061,1.458853,2.606934,34.889127,1.078113,2025.0
min,1.000000,2025-01-02 00:00:00,1.500000e-01,1.000000,75001.0,101.000000,1.000000,0.000000,1.000000,0.000000,0.000000,2025.0
25%,1.000000,2025-03-31 00:00:00,2.521488e+05,11.000000,75010.0,110.000000,18.000000,1.000000,2.000000,0.000000,0.000000,2025.0
50%,1.000000,2025-07-08 00:00:00,4.805000e+05,27.000000,75015.0,115.000000,40.000000,1.000000,3.000000,11.000000,0.000000,2025.0
75%,1.000000,2025-10-07 00:00:00,1.058025e+06,65.000000,75017.0,117.000000,70.000000,2.000000,3.000000,45.000000,2.000000,2025.0
max,5.000000,2025-12-31 00:00:00,6.950000e+08,405.000000,75020.0,120.000000,786.000000,72.000000,4.000000,22372.000000,20.000000,2025.0
std,0.085829,NaN,2.115833e+07,55.363126,4.773738,4.773738,41.919204,1.099105,0.605767,168.139034,1.476392,0.0


In [ ]:
# Summary statistics on key analytical columns only
cols_to_describe = ["Valeur fonciere", "Surface reelle bati", "Nombre pieces principales", "Nombre de lots"]
df_paris[cols_to_describe].describe()

,Valeur fonciere,Surface reelle bati,Nombre pieces principales,Nombre de lots
count,8.275200e+04,82752.000000,82752.000000,82752.000000
mean,3.844312e+06,34.889127,1.078113,1.458853
std,2.115833e+07,168.139034,1.476392,1.099105
min,1.500000e-01,0.000000,0.000000,0.000000
25%,2.521488e+05,0.000000,0.000000,1.000000
50%,4.805000e+05,11.000000,0.000000,1.000000
75%,1.058025e+06,45.000000,2.000000,2.000000
max,6.950000e+08,22372.000000,20.000000,72.000000


#### Examine unexpected extreme values to determine if they should be considered outliers.

In [ ]:
# import dataviz libraries
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Modalities for "Type local"
print(df_paris["Type local"].value_counts())

Type local
Dépendance                                  40171
Appartement                                 37269
Local industriel. commercial ou assimilé     5122
Maison                                        190
Name: count, dtype: int64


In [ ]:
# Re-examine for one space type ("Type local")
# Enter type here !
space_type = "Appartement"

print("-" * 55)
print(f"Distributions of key variables for Space Type: {space_type}")
print("-" * 55)

df_paris_type = df_paris[df_paris['Type local'] == space_type]

for col in cols_to_describe:
    plt.figure(figsize=(8, 4))
    sns.boxplot(y=df_paris_type[col], whis=1.5, color="orange")
    plt.title(f'Distribution of {col}')
    plt.ylabel(col.replace('_', ' ').title()) # Clean up column names for better labels
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
# Setting the space type
space_type = "Appartement"

# Calculate upper and lower bounds for key columns by 'Type local'
print(f"--- Outlier Bounds for space type: {space_type} ---")

df_subset = df_paris[df_paris['Type local'] == space_type]

for col in cols_to_describe:
        # Ensure the column is numeric and not empty for calculation
  if pd.api.types.is_numeric_dtype(df_subset[col]) and not df_subset[col].empty:
    Q1 = df_subset[col].quantile(0.25)
    Q3 = df_subset[col].quantile(0.75)
    IQR = Q3 - Q1

    upper_bound = Q3 + 1.5 * IQR
    if Q1 - 1.5 * IQR < 0:
      lower_bound = 0
    else:
      lower_bound = Q1 - 1.5 * IQR

    print(f"  {col}:")
    print(f"    Lower Bound = {lower_bound:.2f}")
    print(f"    Upper Bound = {upper_bound:.2f}")
  else:
    print(f"  {col}: (Not applicable or data not numeric)")


--- Outlier Bounds for space type: Appartement ---
  Valeur fonciere:
    Lower Bound = 0.00
    Upper Bound = 1872500.00
  Surface reelle bati:
    Lower Bound = 0.00
    Upper Bound = 124.50
  Nombre pieces principales:
    Lower Bound = 0.00
    Upper Bound = 6.00
  Nombre de lots:
    Lower Bound = 0.00
    Upper Bound = 3.50


In [ ]:
# Testing various values to have a closer look at property value
# Setting values to test
type_local= 'Appartement'
upper_bound = 600000000

#Displaying results
print(f" ------ Values above {upper_bound}: ------")
pvalue_outliers = df_paris[(df_paris['Valeur fonciere'] > upper_bound) & (df_paris['Type local'] == type_local)]

display(pvalue_outliers.head(5))

 ------ Values above 600000000: ------


,No disposition,Date mutation,Nature mutation,Valeur fonciere,No voie,Type de voie,Code voie,Voie,Code postal,Commune,Code departement,Code commune,Section,No plan,Nombre de lots,Code type local,Type local,Surface reelle bati,Nombre pieces principales,year
3708872,1,2025-11-19,Vente,695000000.0,2.0,AV,8059,RAYMOND POINCARE,75016,PARIS 16,75,116,FR,4,0,2.0,Appartement,22.0,1.0,2025
3708873,1,2025-11-19,Vente,695000000.0,2.0,AV,8059,RAYMOND POINCARE,75016,PARIS 16,75,116,FR,4,0,2.0,Appartement,74.0,3.0,2025
3708874,1,2025-11-19,Vente,695000000.0,2.0,AV,8059,RAYMOND POINCARE,75016,PARIS 16,75,116,FR,4,0,2.0,Appartement,90.0,4.0,2025
3708875,1,2025-11-19,Vente,695000000.0,2.0,AV,8059,RAYMOND POINCARE,75016,PARIS 16,75,116,FR,4,0,2.0,Appartement,90.0,3.0,2025
3708876,1,2025-11-19,Vente,695000000.0,2.0,AV,8059,RAYMOND POINCARE,75016,PARIS 16,75,116,FR,4,0,2.0,Appartement,90.0,4.0,2025


Preview of the distribution of the main variables for the type `Appartement`. Y-limits are set in the area of the upper-bounds to have a better visual of the distribution.

In [ ]:
df_paris_type = df_paris[df_paris['Type local'] == "Appartement"]

fix, ax = plt.subplots(2, 2, figsize=(12, 6))

sns.histplot(df_paris_type['Nombre pieces principales'], ax=ax[0, 0])
ax[0, 0].set_xlim(xmin=0, xmax=10)
ax[0, 0].set_title("Number of rooms")

sns.histplot(df_paris_type['Valeur fonciere'], ax=ax[0,1])
ax[0, 1].set_xlim(xmin=0, xmax=2000000)
ax[0, 1].set_title("Property value")

sns.histplot(df_paris_type['Surface reelle bati'], ax=ax[1,0])
ax[1, 0].set_xlim(xmin=0, xmax=150)
ax[1, 0].set_title("Surface area")

sns.histplot(df_paris_type['Nombre de lots'], ax=ax[1,1])
ax[1,1].set_xlim(xmin=0, xmax=4)
ax[1,1].set_title("Number of lots")

plt.subplots_adjust(hspace=0.5);


Findings:
* We can expect the distribution for key variables like property value, surface area, number of lots, and number of rooms to vary by property type (House, Appartment, Commercial/industrial space, outbuilding).
* The distribution of key variables is generally positively skewed, meaning it has a long right tail.
* Property / transaction value is listed for an entire transaction across all "items" belonging to the same transaction ("No disposition").
* The number of items in the transaction is NOT reflected by the variable "Nombre de lots".

Results:
* We will want to treat outliers the data in the context of the corresponding space type.
* It may be necessary to group rows to find out the value/m^2 metric

### 6. Renaming variables

To ease the analysis process we replace the French column names, transaction types, and property types with English translations.

In [ ]:
# Create renaming dictionary for the columns
rename_dict = {
    "No disposition": "transaction_number",
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "Type de voie": "street_type",
    "Code voie": "street_code",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "commune",
    "Code departement": "department_code",
    "Code commune": "commune_code",
    "Section": "section",
    "No plan": "plot_number",
    "Nombre de lots": "lot_count",
    "Code type local": "property_type_code",
    "Type local": "property_type",
    "Surface reelle bati": "surface_area",
    "Nombre pieces principales": "room_count"
}

# Apply to dataframe
df_paris = df_paris.rename(columns=rename_dict)

# Check column names
display(df_paris.info())

<class 'pandas.core.frame.DataFrame'>
Index: 82752 entries, 3630389 to 3714828
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   transaction_number  82752 non-null  int64         
 1   transaction_date    82752 non-null  datetime64[ns]
 2   transaction_type    82752 non-null  object        
 3   property_value      82752 non-null  float64       
 4   street_number       82752 non-null  float64       
 5   street_type         82752 non-null  object        
 6   street_code         82752 non-null  object        
 7   street_name         82752 non-null  object        
 8   postal_code         82752 non-null  Int64         
 9   commune             82752 non-null  object        
 10  department_code     82752 non-null  object        
 11  commune_code        82752 non-null  int64         
 12  section             82752 non-null  object        
 13  plot_number         82752 non-null  int64  

None

Renaming the modalities in `transaction_types`.

In [ ]:
# Create French-English dictionary for transaction types
transaction_types = {
    "Vente": "Sale",
    "Vente en l'état futur d'achèvement": "Off-plan sale",
    "Echange": "Exchange",
    "Adjudication": "Auction sale",
    "Vente terrain à bâtir": "Building lot sale"}
print("Translations for transaction types:\n", transaction_types)

# Replace with translations
df_paris['transaction_type'] = df_paris['transaction_type'].replace(transaction_types)

# Check new values
print("Translated values:\n")
display(df_paris['transaction_type'].value_counts())

Translations for transaction types:
 {'Vente': 'Sale', "Vente en l'état futur d'achèvement": 'Off-plan sale', 'Echange': 'Exchange', 'Adjudication': 'Auction sale', 'Vente terrain à bâtir': 'Building lot sale'}
Translated values:



,count
transaction_type,
Sale,82044
Exchange,477
Auction sale,154
Off-plan sale,75
Building lot sale,2


Renaming the modalities in `property_types`.

In [ ]:
# Create French-English dictionary for property types
property_types = {
    "Dépendance": "Outbuilding",
    "Appartement": "Apartment",
    "Local industriel. commercial ou assimilé": "Industrial, commercial, or similar",
    "Maison": "House"
    }

# Replace with the English translations
df_paris['property_type'] = df_paris['property_type'].replace(property_types)

# Check new values
print("Translated values:\n")
display(df_paris['property_type'].value_counts())

Translated values:



,count
property_type,
Outbuilding,40171
Apartment,37269
"Industrial, commercial, or similar",5122
House,190


Verify that each modality in `property_type` matches with the expected code and there are no crossover values.

In [ ]:
# Define the expected mapping between property_type and property_type_code
expected_mapping = {
    'Apartment': 2.0,
    'House': 1.0,
    'Industrial, commercial, or similar': 4.0,
    'Outbuilding': 3.0
}

# Create a new column with the expected property_type_code based on property_type
df_paris['expected_property_type_code'] = df_paris['property_type'].map(expected_mapping)

# Check for discrepancies
discrepancies = df_paris[df_paris['property_type_code'] != df_paris['expected_property_type_code']]

# Print results
if discrepancies.empty:
    print("All records show consistent mapping between 'property_type' and 'property_type_code'.")
else:
    print(f"Found {len(discrepancies)} discrepancies between 'property_type' and 'property_type_code':")
    display(discrepancies[['property_type', 'property_type_code', 'expected_property_type_code']].head())

# Drop the temporary 'expected_property_type_code' column
df_paris = df_paris.drop(columns=['expected_property_type_code'])

All records show consistent mapping between 'property_type' and 'property_type_code'.


### Cleaning Summary



In [ ]:
# Snapshots captured at each step ensure row/column counts at each step of the pipeline.

summary = pd.DataFrame([
    {"Stage":         "Raw (all France)",
     "Rows":          snap_raw_rows,
     "Columns":       snap_raw_cols,
     "Missing cells": snap_raw_missing},

    {"Stage":         "After Step 2: Paris filter",
     "Rows":          snap_paris_rows,
     "Columns":       snap_paris_cols,
     "Missing cells": snap_paris_missing},

    {"Stage":         "After Step 3: Drop sparse columns (>50%)",
     "Rows":          snap_drop_cols_rows,
     "Columns":       snap_drop_cols_cols,
     "Missing cells": snap_drop_cols_missing},

    {"Stage":         "After Step 4: Drop rows with missing values",
     "Rows":          len(df_paris),
     "Columns":       df_paris.shape[1],
     "Missing cells": int(df_paris.isna().sum().sum())},
])

display(summary.style.hide(axis="index"))
print(f"\n-Columns reduced : {snap_raw_cols} -> {df_paris.shape[1]}")
print(f"- Rows after Paris filter : {snap_paris_rows:,} of {snap_raw_rows:,} ({snap_paris_rows/snap_raw_rows*100:.1f}%)")
print(f"- Rows after final dropna : {len(df_paris):,} of {snap_paris_rows:,} ({len(df_paris)/snap_paris_rows*100:.1f}% retained)")

Stage,Rows,Columns,Missing cells
Raw (all France),3714829,44,90144608
After Step 2: Paris filter,84252,44,1842337
After Step 3: Drop sparse columns (>50%),84252,21,12065
After Step 4: Drop rows with missing values,82752,20,0



-Columns reduced : 44 -> 20
- Rows after Paris filter : 84,252 of 3,714,829 (2.3%)
- Rows after final dropna : 82,752 of 84,252 (98.2% retained)


## Export Cleaned Dataset

In [ ]:
df_paris.head()

,transaction_number,transaction_date,transaction_type,property_value,street_number,street_type,street_code,street_name,postal_code,commune,department_code,commune_code,section,plot_number,lot_count,property_type_code,property_type,surface_area,room_count,year
3425131,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020,PARIS 20,75,120,BM,133,2,2.0,Apartment,86.0,4.0,2024
3425132,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020,PARIS 20,75,120,BM,133,2,3.0,Outbuilding,0.0,0.0,2024
3425133,1,2024-01-04,Sale,1042000.0,16.0,RUE,2786,DE LA DHUIS,75020,PARIS 20,75,120,BM,133,1,3.0,Outbuilding,0.0,0.0,2024
3425134,1,2024-01-04,Sale,1042000.0,16.0,RUE,2786,DE LA DHUIS,75020,PARIS 20,75,120,BM,133,1,3.0,Outbuilding,0.0,0.0,2024
3425135,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020,PARIS 20,75,120,BM,133,2,3.0,Outbuilding,0.0,0.0,2024


In [ ]:
# Save the cleaned dataset

output_path = "../data/dvf_paris_2024_2025.csv"
df_paris.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}")
print(f"   Shape : {df_paris.shape[0]:,} rows x {df_paris.shape[1]} columns")

Saved: dvf_paris_2024_2025.csv
   Shape : 156,798 rows x 20 columns
